# OOP Principles in Python
### Object-Oriented Programming

In this notebook we will cover:
1. **Access Modifiers** (Public - Protected - Private)
2. **Class Variables vs Instance Variables**
3. **Encapsulation**
4. **Inheritance**
5. **Polymorphism**
6. **Abstraction**


## 1) Access Modifiers

In languages like Java or C++ there are explicit keywords like `public`,
`private`, and `protected`. Python does **not** have real keywords for
this — instead there is a **naming convention** developers agree to follow:

| Type | Syntax | Meaning |
|---|---|---|
| **Public** | `name` | Accessible from anywhere, inside or outside the class |
| **Protected** | `_name` (single underscore) | "Use only inside the class or subclasses" — just a convention, not enforced |
| **Private** | `__name` (double underscore) | Python actually applies **Name Mangling**, which makes direct access harder |

⚠️ **Important:** in Python, "private" is not exactly like Java's private.
It's actually "hard to access" rather than "impossible to access" — there is
a way to reach it anyway (we'll see it below), but the convention says you
should never do that in real code.

In [1]:
class Employee:
    def __init__(self, name, salary, ssn):
        self.name = name          # Public: anyone can access and modify it freely
        self._salary = salary     # Protected: a moral warning only - "don't touch from outside"
        self.__ssn = ssn          # Private: Python applies Name Mangling to it 
        self.salary = 556

emp = Employee("Ahmed", 15000, "29012345678")

print(emp.name)        # public -> works normally
print(emp._salary)     # protected -> still works! but it's bad practice to access it directly
print(emp.__ssn)     


Ahmed
15000


AttributeError: 'Employee' object has no attribute '__ssn'

In [27]:
print(emp.salary is emp._salary)  # True, both refer to the same value

False


In [2]:
# Try to access the private attribute directly
try:
    print(emp.__ssn)
except AttributeError as e:
    print("Error:", e)


Error: 'Employee' object has no attribute '__ssn'


**Why did we get an Error?**
Python does something called **Name Mangling**: any attribute whose name
starts with `__` (and does not end with `__`) is automatically renamed to
`_ClassName__attribute`.

So `self.__ssn` inside `Employee` actually becomes `self._Employee__ssn`.

In [8]:
# This way you CAN reach it - but you should never do this in real code
print(emp._Employee__ssn)

# This is exactly the idea behind Encapsulation, which we'll cover in detail
# next: protecting data and controlling access to it through methods instead
# of accessing it directly


29012345678


### Access Modifiers - Summary

- **Public** (`name`): use it for data that's fine for anyone to see/modify
- **Protected** (`_name`): use it for internal data that subclasses need to
  access, but outside code shouldn't touch
- **Private** (`__name`): use it for sensitive data where you want full
  control over access, only through methods (getters/setters)

## 2) Class Variables vs Instance Variables

### The key question: when does a variable belong to the Class, and when does it belong to the Object?

- **Instance Variable (belongs to the Object):**
  - Defined inside `__init__` (or any method) using `self.name = ...`
  - **Every object** has its own independent copy of it
  - Changing it on one object does NOT affect other objects

- **Class Variable (belongs to the Class):**
  - Defined **directly inside the class body, outside any method**
  - **All objects share it** — same value, same place in memory
  - Changing it from the class itself affects all objects, old and new

In [ ]:
class Student:
    # Class Variable: shared across all objects
    school_name = "AI Institute"
    total_students = 0   # a shared counter for the number of students

    def __init__(self, name, grade):
        # Instance Variables: each object has its own independent copy
        self.name = name
        self.grade = grade
        self.school_name = "test value"
        Student.total_students += 1   # every time we create a new object, the shared counter increases


s1 = Student("Mohamed", "A")
s2 = Student("Sara", "B")

print(s1.name, "-", s1.school_name)
print(s2.name, "-", s2.school_name)
print("Total students:", Student.total_students)


Mohamed - AI Institute
Sara - AI Institute
Total students: 2


In [10]:
# If we change the class variable from the class itself -> all objects are affected
Student.school_name = "Advanced Python Academy"

print(s1.school_name)   # changed
print(s2.school_name)   # changed too - because it's a shared variable


# But if we change an instance variable on one object only -> the others are NOT affected
s1.grade = "A+"
print(s1.grade)   # A+
print(s2.grade)   # still B - because grade is an instance variable, specific to each object


Advanced Python Academy
Advanced Python Academy
A+
B


⚠️ **A common trap:** if you do `s1.school_name = "something else"` (from
the object, not the class), Python will create a **new instance variable
called school_name that belongs to s1 only**, and it will shadow the class
variable for s1 alone, without affecting s2 or the class itself. Try it
below and see the difference.

In [11]:
s2.school_name = "test value"   # this creates a new instance variable for s2 only

print(s1.school_name)          # still the shared value
print(s2.school_name)          # its own new value
print(Student.school_name)     # the class value itself is still unchanged


Advanced Python Academy
test value
Advanced Python Academy


### A simple rule to remember it by
- Use an **Instance Variable** for data that differs from one object to
  another (e.g. student name, grade)
- Use a **Class Variable** for data that is constant or shared across all
  objects (e.g. school name, a global counter, fixed settings)

## 3) Encapsulation

Idea: hide the data (attributes) inside the class and don't let anyone
modify it directly from outside — instead, control access to it through
methods (getters/setters).

In [12]:
class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        self.__balance = balance   # private attribute

    def deposit(self, amount):
        if amount > 0:
            self.__balance += amount
            print(f"Deposited {amount}. Current balance: {self.__balance}")
        else:
            print("Amount must be greater than zero")

    def withdraw(self, amount):
        if amount > self.__balance:
            print("Insufficient balance")
        else:
            self.__balance -= amount
            print(f"Withdrew {amount}. Current balance: {self.__balance}")

    def get_balance(self):
        return self.__balance


account = BankAccount("Ahmed", 1000)
account.deposit(500)
account.withdraw(2000)
print("Final balance:", account.get_balance())


Deposited 500. Current balance: 1500
Insufficient balance
Final balance: 1500


## 4) Inheritance

Idea: a new class (Child) takes on attributes and behaviors from an
existing class (Parent), so we can reuse code instead of duplicating it.

In [32]:
class Animal:
    def __init__(self, name):
        self.name = name

    def eat(self):
        print(f"{self.name} is eating")

    def make_sound(self):
        print(f"{self.name} makes a sound")


class Dog(Animal):          # Dog inherits from Animal
    def make_sound(self):   # Overriding the method
        print(f"{self.name} barks: Woof woof")


class Cat(Animal):
    def __init__(self, name):
        super().__init__(name)  # Call the parent class constructor
    def __init__(self, name ,age):
        super().__init__(name)
        self.age = age

    def make_sound(self):
        print(f"{self.name} meows: Meow , I am {self.age} years old!" )


dog = Dog("Rex")
cat = Cat("Whiskers", 3)
dog.eat()     # method inherited from the parent class Animal
cat.eat()
dog.make_sound()
cat.make_sound()


Rex is eating
Whiskers is eating
Rex barks: Woof woof
Whiskers meows: Meow , I am 3 years old!


## 5) Polymorphism

Idea: the same method name behaves differently depending on the type of
the object calling it, without needing to know its exact type in advance.

In [14]:
def animal_sound(animal: Animal):
    animal.make_sound()   # same line of code, but the result differs based on the type of animal


animals = [Dog("Rex"), Cat("Whiskers")]
for a in animals:
    animal_sound(a)


Rex barks: Woof woof
Whiskers meows: Meow


## 6) Abstraction

Idea: define a "general shape" (interface) that child classes must follow,
without specifying the internal implementation details of each one.

In [15]:
from abc import ABC, abstractmethod

class Shape(ABC):           # an abstract class - cannot be instantiated directly
    @abstractmethod
    def area(self):
        pass

    @abstractmethod
    def perimeter(self):
        pass


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return round(3.14 * self.radius ** 2, 2)

    def perimeter(self):
        return round(2 * 3.14 * self.radius, 2)


shapes = [Rectangle(4, 5), Circle(3)]
for s in shapes:
    print(f"{type(s).__name__} -> Area: {s.area()}, Perimeter: {s.perimeter()}")


Rectangle -> Area: 20, Perimeter: 18
Circle -> Area: 28.26, Perimeter: 18.84


In [16]:
# Try to create an object directly from the abstract class
try:
    shape = Shape()
except TypeError as e:
    print("Error:", e)


Error: Can't instantiate abstract class Shape without an implementation for abstract methods 'area', 'perimeter'


## Quick Summary

| Principle | One-line idea |
|---|---|
| Access Modifiers | Who can access the data (public / protected / private) |
| Class vs Instance Variables | A variable shared by all objects, or specific to each object |
| Encapsulation | Hiding data and protecting it from direct modification |
| Inheritance | Reusing code from a parent class |
| Polymorphism | Same call, different behavior depending on the object's type |
| Abstraction | Enforcing a general interface without implementation details |
